# Active Learning con LightGBM

**Objetivo:** Usar los labels de severidad generados por el LLM como supervisión débil para entrenar LightGBM y compararlo con LOF.

**Flujo:**
1. Cargar anomalías detectadas por LOF (V1)
2. Generar labels via LLM Summarizer (o heurística si Ollama no disponible)
3. Entrenar LightGBM con esos labels
4. Comparar LOF vs LightGBM: F1, Precision, Recall, AUPR
5. Feature importance del modelo supervisado
6. Registrar experimento en MLflow

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from datetime import timedelta
from pathlib import Path

from src.data.loader import load_bgl_logs
from src.data.preprocessor import add_severity_score, train_test_split_temporal
from src.features.engineering import load_features
from src.models.detector import AnomalyDetector
from src.models.evaluator import evaluate, compare_models, get_pr_curve
from src.models.active_learner import ActiveLearner, encode_llm_labels
from src.llm.summarizer import LogSummarizer, build_anomaly_context

plt.rcParams['figure.dpi'] = 120

FEATURES_PATH = Path('../data/processed/features_train.parquet')
LABELS_PATH   = Path('../data/labels/llm_confirmed.parquet')
LOF_PATH      = Path('../models/saved/lof_v1.joblib')
AL_PATH       = Path('../models/saved/active_learner_v1.joblib')
MLFLOW_URI    = '../mlflow'

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('anomaly-detection-v2')

## 1. Cargar features y modelo LOF

In [ ]:
feat_df = load_features(FEATURES_PATH)
feature_cols = [c for c in feat_df.columns if c not in {'timestamp', 'node', 'is_anomaly'}]
train_df, test_df = train_test_split_temporal(feat_df, test_fraction=0.2)

lof = AnomalyDetector.load(LOF_PATH)
X_train = train_df[feature_cols].fillna(0).values
X_test  = test_df[feature_cols].fillna(0).values
y_test  = test_df['is_anomaly'].values

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Anomalías en test: {y_test.sum():,} ({y_test.mean():.1%})')

## 2. Generar labels LLM (o heurística como fallback)

In [ ]:
summarizer = LogSummarizer(model='llama3.2')
print(f'Ollama disponible: {summarizer.is_available}')

# Anomalías detectadas en train set — tomar las top 300 por score
y_train_pred = lof.predict(X_train)
scores_train = lof.score_samples(X_train)
train_df = train_df.copy()
train_df['_pred'] = y_train_pred
train_df['_score'] = scores_train
train_anomalies = train_df[train_df['_pred'] == 1].nlargest(300, '_score').reset_index(drop=True)
print(f'Candidatos para labeling: {len(train_anomalies)}')

In [ ]:
if LABELS_PATH.exists():
    print('Cargando labels existentes...')
    labels_df = pd.read_parquet(LABELS_PATH)
    print(f'Labels: {len(labels_df)} | distribución: {labels_df["llm_label"].value_counts().to_dict()}')

elif summarizer.is_available:
    print('Generando labels con LLM (puede tardar ~3 min)...')
    df_raw = load_bgl_logs(Path('../data/raw/BGL.log'), nrows=400_000)
    df_raw = add_severity_score(df_raw)
    df_train_raw, _ = train_test_split_temporal(df_raw)

    records = []
    for i, row in train_anomalies.iterrows():
        ts = row['timestamp']
        node = str(row['node'])
        window = df_train_raw[
            (df_train_raw['timestamp'] >= ts - timedelta(minutes=5)) &
            (df_train_raw['timestamp'] <= ts + timedelta(minutes=5))
        ]
        if len(window) == 0:
            continue
        ctx = build_anomaly_context(window, anomaly_score=float(row['_score']), node=node, timestamp=str(ts))
        result = summarizer.summarize(ctx)
        sev = result.get('severidad', 'UNKNOWN')
        encoded = encode_llm_labels(pd.Series([sev]))
        if not pd.isna(encoded.iloc[0]):
            records.append({'timestamp': ts, 'node': node, 'severidad': sev, 'llm_label': int(encoded.iloc[0])})
        if (i + 1) % 50 == 0:
            print(f'  {i+1}/{len(train_anomalies)} procesados...')

    labels_df = pd.DataFrame(records)
    LABELS_PATH.parent.mkdir(parents=True, exist_ok=True)
    labels_df.to_parquet(LABELS_PATH, index=False)
    print(f'Labels guardados: {len(labels_df)} | distribución: {labels_df["llm_label"].value_counts().to_dict()}')

else:
    print('Ollama no disponible — usando ground truth is_anomaly como fallback')
    heuristic = train_df[['timestamp', 'node', 'is_anomaly']].copy()
    heuristic['llm_label'] = heuristic['is_anomaly'].astype(int)
    heuristic['severidad'] = heuristic['llm_label'].map({1: 'CRITICAL', 0: 'LOW'})
    labels_df = heuristic[['timestamp', 'node', 'severidad', 'llm_label']]
    LABELS_PATH.parent.mkdir(parents=True, exist_ok=True)
    labels_df.to_parquet(LABELS_PATH, index=False)
    print(f'Labels heurísticos: {len(labels_df)} | distribución: {labels_df["llm_label"].value_counts().to_dict()}')

## 3. Entrenar LightGBM con labels generados

In [ ]:
merged = train_df.merge(labels_df[['timestamp', 'node', 'llm_label']], on=['timestamp', 'node'], how='inner')
merged = merged.dropna(subset=['llm_label'])
print(f'Muestras con label: {len(merged):,} | ratio positivos: {merged["llm_label"].mean():.1%}')

X_train_al = merged[feature_cols].fillna(0).values
y_train_al = merged['llm_label'].values.astype(int)

learner = ActiveLearner(n_estimators=300, learning_rate=0.05, max_depth=6)
learner.fit(X_train_al, y_train_al, feature_names=feature_cols)
AL_PATH.parent.mkdir(parents=True, exist_ok=True)
learner.save(AL_PATH)
print(f'Modelo guardado: {AL_PATH}')

## 4. Comparativa LOF vs LightGBM

In [ ]:
y_pred_lof = lof.predict(X_test)
scores_lof = lof.score_samples(X_test)
result_lof = evaluate(y_test, y_pred_lof, scores_lof, model_name='LOF (V1)')

y_pred_al = learner.predict(X_test)
scores_al = learner.score_samples(X_test)
result_al = evaluate(y_test, y_pred_al, scores_al, model_name='LightGBM (Active Learning)')

comparison = compare_models([result_lof, result_al])
print('=== Comparativa LOF vs LightGBM Active Learning ===')
print(comparison.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for result, scores, color, ls in [
    (result_lof, scores_lof, '#4361ee', '-'),
    (result_al,  scores_al,  '#ef233c', '--'),
]:
    prec, rec, _ = get_pr_curve(y_test, scores)
    ax.plot(rec, prec, color=color, linewidth=2, linestyle=ls,
            label=f'{result.model_name} (AUPR={result.aupr:.3f}, F1={result.f1:.3f})')

baseline = y_test.mean()
ax.axhline(y=baseline, color='gray', linestyle=':', alpha=0.6, label=f'Baseline aleatorio ({baseline:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall — LOF vs LightGBM Active Learning')
ax.legend()
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig('../reports/figures/06_pr_curves_al.png', dpi=150)
plt.show()

## 5. Feature Importance — LightGBM vs LOF

In [ ]:
imp = learner.get_feature_importance().head(15)

fig, ax = plt.subplots(figsize=(9, 5))
imp.sort_values().plot(kind='barh', ax=ax, color='#4361ee')
ax.set_title('Top 15 features — LightGBM Active Learning')
ax.set_xlabel('Importancia (gain)')
plt.tight_layout()
plt.savefig('../reports/figures/06_feature_importance_al.png', dpi=150)
plt.show()

print('Top 10 features:')
print(imp.head(10).to_string())

## 6. Registrar experimento en MLflow

In [ ]:
with mlflow.start_run(run_name='active_learner_v1'):
    mlflow.log_params(learner.get_params())
    mlflow.log_params({
        'n_labeled_samples': len(merged),
        'label_positive_rate': round(float(merged['llm_label'].mean()), 4),
        'feature_count': len(feature_cols),
    })
    mlflow.log_metrics({
        'f1':              round(result_al.f1, 4),
        'precision':       round(result_al.precision, 4),
        'recall':          round(result_al.recall, 4),
        'aupr':            round(result_al.aupr, 4),
        'f1_lof_baseline': round(result_lof.f1, 4),
    })
    mlflow.sklearn.log_model(learner._model, 'lgbm_model')
    for fig_path in ['../reports/figures/06_pr_curves_al.png', '../reports/figures/06_feature_importance_al.png']:
        if Path(fig_path).exists():
            mlflow.log_artifact(fig_path)
    run_id = mlflow.active_run().info.run_id

print(f'Run registrado: {run_id}')
print(f'Ver: mlflow ui --backend-store-uri ../mlflow  →  http://localhost:5000')
print()
print('Resumen final:')
print(f'  LOF (V1)              F1={result_lof.f1:.4f}  AUPR={result_lof.aupr:.4f}')
print(f'  LightGBM (Active AL)  F1={result_al.f1:.4f}  AUPR={result_al.aupr:.4f}')